# 2. Основы тензоров PyTorch

**Цель:** Изучить создание тензоров, операции, типы данных, устройства и автоматическое дифференцирование.

---

In [ ]:
import sys
import os
import logging

LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("tensors")

import torch
import numpy as np
log.info("PyTorch %s loaded", torch.__version__)

## 2.1 Создание тензоров

In [ ]:
log.debug("Creating tensors from different sources")

# Из списка
t1 = torch.tensor([1, 2, 3, 4, 5])
print(f"From list: {t1}, shape={t1.shape}")

# Из numpy
t2 = torch.tensor(np.array([[1, 2], [3, 4]]))
print(f"From numpy:\n{t2}")

# Случайный
t3 = torch.randn(2, 3)
print(f"Random normal:\n{t3}")

# Нули / единицы
t4 = torch.zeros(2, 4)
t5 = torch.ones(3, 3)
t6 = torch.eye(4)
print(f"Zeros:\n{t4}")
print(f"Ones:\n{t5}")
print(f"Identity:\n{t6}")

# Arange / linspace
t7 = torch.arange(0, 10, 2)
t8 = torch.linspace(0, 1, 5)
print(f"Arange: {t7}")
print(f"Linspace: {t8}")

## 2.2 Типы данных (dtype) и устройства (device)

In [ ]:
log.debug("Exploring dtypes and devices")

# Типы данных
float_t = torch.tensor([1.0, 2.0], dtype=torch.float32)
double_t = torch.tensor([1.0, 2.0], dtype=torch.float64)
int_t = torch.tensor([1, 2], dtype=torch.int64)
print(f"float32: {float_t.dtype}")
print(f"float64: {double_t.dtype}")
print(f"int64: {int_t.dtype}")

# Приведение типа
x = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"Default: {x.dtype}")

# Перемещение на устройство
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
t_gpu = torch.tensor([1, 2, 3], device=device)
print(f"On device: {t_gpu.device}")
log.info("Using device: %s", device)

## 2.3 Индексация и срезы

In [ ]:
log.debug("Indexing and slicing")

x = torch.randn(4, 5)
print(f"Tensor 4x5:\n{x}\n")

print(f"First row: {x[0]}")
print(f"First column: {x[:, 0]}")
print(f"Submatrix (1:3, 1:4):\n{x[1:3, 1:4]}")
print(f"Element (2, 3): {x[2, 3]}")

# Boolean indexing
mask = x > 0
print(f"Positive values: {x[mask]}")
print(f"Number of positive: {mask.sum()}")

## 2.4 Решейп и изменение формы

In [ ]:
log.debug("Reshape operations")

x = torch.arange(12)
print(f"Original: {x.shape} -> {x}")

# view (разделяет память)
y = x.view(3, 4)
print(f"View (3,4):\n{y}")

# reshape (может копировать)
z = x.reshape(2, 6)
print(f"Reshape (2,6):\n{z}")

# Flatten
w = torch.randn(2, 3, 4)
print(f"Flatten {w.shape} -> {w.flatten().shape}")

# Transpose
t = torch.randn(2, 3)
print(f"Transpose {t.shape} -> {t.T.shape}")

# Unsqueeze / squeeze
a = torch.randn(3)
print(f"Unsqueeze {a.shape} -> {a.unsqueeze(0).shape} -> {a.unsqueeze(1).shape}")
b = torch.randn(1, 3, 1, 4)
print(f"Squeeze {b.shape} -> {b.squeeze().shape}")

## 2.5 Математические операции

In [ ]:
log.debug("Math operations")

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")  # поэлементное
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

# Матричное умножение
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = torch.mm(A, B)  # или A @ B
C2 = A @ B
print(f"Matmul: {A.shape} @ {B.shape} = {C.shape}")

# Batch matmul
batch_A = torch.randn(8, 3, 4)
batch_B = torch.randn(8, 4, 5)
batch_C = torch.bmm(batch_A, batch_B)
print(f"Batch matmul: {batch_C.shape}")

## 2.6 Broadcasting

In [ ]:
log.debug("Broadcasting demonstration")

# Скаляр + тензор
x = torch.ones(3, 4)
y = x + 5
print(f"Tensor + scalar:\n{y}")

# Разные формы
a = torch.randn(3, 1)  # столбец
b = torch.randn(4)     # строка
c = a + b              # broadcasting: (3,1) + (4,) -> (3,4)
print(f"Broadcast {a.shape} + {b.shape} -> {c.shape}")

# Практика: нормализация батча вручную
batch = torch.randn(16, 64)  # (batch, features)
mean = batch.mean(dim=0, keepdim=True)  # (1, 64)
std = batch.std(dim=0, keepdim=True)    # (1, 64)
normalized = (batch - mean) / std
print(f"Manual batch norm: input {batch.shape}, output {normalized.shape}")
print(f"Normalized mean: {normalized.mean():.4f}, std: {normalized.std():.4f}")

## 2.7 Автоматическое дифференцирование (Autograd)

In [ ]:
log.debug("Autograd basics")

# Простой граф вычислений
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x: {x}, requires_grad: {x.requires_grad}")

y = x ** 2 + 2 * x + 1
print(f"y = x^2 + 2x + 1: {y}")

z = y.sum()
print(f"z = sum(y): {z}")

z.backward()
print(f"dz/dx: {x.grad}")  # dy/dx = 2x + 2 -> [6.0, 8.0]

In [ ]:
log.debug("Autograd on chain rule")

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x ** 2 * y + y ** 3
f.backward()

print(f"x = {x.item()}, y = {y.item()}")
print(f"f(x,y) = x^2*y + y^3 = {f.item()}")
print(f"df/dx = 2*x*y = {x.grad.item():.2f} (expected: {2*2*3:.2f})")
print(f"df/dy = x^2 + 3*y^2 = {y.grad.item():.2f} (expected: {4 + 27:.2f})")

In [ ]:
log.debug("Gradient accumulation and zeroing")

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Первый backward
loss = (x ** 2).sum()
loss.backward()
print(f"After first backward: {x.grad}")  # 2*x

# Градиенты накапливаются!
loss2 = (x * 2).sum()
loss2.backward()
print(f"After second backward (ACCUMULATED): {x.grad}")  # 2*x + 2

# Нужно обнулять
x.grad.zero_()
print(f"After zero_(): {x.grad}")

In [ ]:
log.debug("detach() and no_grad()")

x = torch.tensor([2.0], requires_grad=True)

# detach() — отсоединить от графа
y = x ** 2
z = y.detach()  # z не требует градиента
w = z ** 2
print(f"z.requires_grad: {z.requires_grad}")

# torch.no_grad() — контекст без градиентов
with torch.no_grad():
    y_no_grad = x ** 3
    print(f"y_no_grad.requires_grad: {y_no_grad.requires_grad}")

print("Detach and no_grad demonstrated")

## 2.8 Функция потерь: MSE вручную vs torch

In [ ]:
log.debug("MSE loss comparison")

pred = torch.tensor([2.5, 0.0, 1.2, -0.8], requires_grad=True)
target = torch.tensor([3.0, -0.5, 1.0, 0.0])

# MSE вручную
mse_manual = ((pred - target) ** 2).mean()
print(f"MSE (manual): {mse_manual.item():.4f}")

# MSE через torch
mse_torch = torch.nn.functional.mse_loss(pred, target)
print(f"MSE (torch):  {mse_torch.item():.4f}")

# Градиенты по MSE
mse_manual.backward()
print(f"grad w.r.t pred: {pred.grad}")
print(f"grad formula: 2*(pred-target)/N = {2*(pred.data - target)/len(pred)}")

## Выводы

In [ ]:
print("=== Tensor fundamentals complete ===")
print("Topics covered:")
print("  - Tensor creation (from list, numpy, random, zeros, ones)")
print("  - Data types (dtype) and devices (device)")
print("  - Indexing, slicing, reshaping (view, reshape, transpose)")
print("  - Math operations and matrix multiplication")
print("  - Broadcasting")
print("  - Autograd: backward, gradients, detach, no_grad")
print("  - Loss functions and gradient computation")
log.info("Tensor fundamentals notebook complete")